# Amazon keyword research with NIMBLE_AGENT_RUN

**Scenario.** You're researching a product category — say, *noise canceling headphones* — and you want a daily snapshot of who is ranking on Amazon for that query, at what price, with what ratings. Not the ASINs *you* care about (you might not have any yet) — the ASINs *Amazon's algorithm* is surfacing for *your customers*.

**What you'll build.**

1. A `KEYWORD_QUERIES` table of research terms (one row per query you want tracked).
2. A `PRODUCT_SEARCH_RESULTS` history table (one row per *product* per query per refresh — many rows per query).
3. A single `INSERT INTO … SELECT … FROM TABLE(NIMBLE_AGENT_RUN('amazon_serp', …)) , LATERAL FLATTEN(...)` statement that runs every query through Nimble's Amazon Search Engine Results Page Web Search Agent and explodes the product array into typed rows.
4. A `V_NEW_ENTRANTS` view that flags ASINs ranking on a tracked query for the first time in the last 7 days — useful for spotting fast-rising competitors.
5. A `schedule.sql` that puts the whole pipeline on a daily Snowflake Task (separate file).

**The "one input row → many output rows" pattern.** A scalar UDF can't express this — it returns one VARIANT per call. A stored procedure can do it, but you pay for a Python middle-tier and lose the inline-SQL ergonomics. `NIMBLE_AGENT_RUN` is a UDTF that yields one row per call; pairing it with `LATERAL FLATTEN(INPUT => a.parsing)` expands the agent's product array into one Snowflake row per result. The whole pipeline stays SQL.

**Prereqs.** Run `../../setup/setup.sql` and `../../udtf-data-feeds/nimble_agent_run.sql` first — they create the role, database, warehouse, secret, external access integration, and the `NIMBLE_AGENT_RUN` UDTF this notebook calls. The Cortex Agent scripts in `../../cortex-agent-tools/` are not needed for this recipe.

**Runtime target.** Plain Jupyter `.ipynb` (nbformat v4). Opens in either Snowflake Notebooks runtime — [Notebooks in Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-overview) (GA Feb 2026, Jupyter-compatible) or Legacy Snowflake Notebooks — and in local JupyterLab against a Snowpark session. See this recipe's `README.md` for import steps.


In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_role("NIMBLE_ROLE")
session.use_warehouse("NIMBLE_AGENT_WH")
session.use_database("NIMBLE_INTEGRATION")
session.use_schema("RECIPES")

session.sql("""
CREATE OR REPLACE TABLE KEYWORD_QUERIES (
    keyword   STRING PRIMARY KEY,
    category  STRING
)
""").collect()

# Sample queries are illustrative placeholders; replace with your own
# category-research terms before running against production.
session.sql("""
INSERT INTO KEYWORD_QUERIES (keyword, category) VALUES
    ('noise canceling headphones', 'Electronics'),
    ('kitchen knife set',          'Home'),
    ('yoga mat',                   'Fitness')
""").collect()

session.sql("""
CREATE OR REPLACE TABLE PRODUCT_SEARCH_RESULTS (
    keyword       STRING,
    category      STRING,
    position      INTEGER,
    asin          STRING,
    title         STRING,
    price         NUMBER(10, 2),
    currency      STRING,
    rating        NUMBER(3, 2),
    review_count  INTEGER,
    image_url     STRING,
    sponsored     BOOLEAN,
    status        STRING,
    enriched_at   TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
)
""").collect()

print("Seeded KEYWORD_QUERIES with 3 rows; PRODUCT_SEARCH_RESULTS ready.")


## The lateral-join + FLATTEN pattern

The Amazon SERP Web Search Agent takes a `keyword` string and returns a JSON object whose `parsing` (the array itself; SERP-style agents return `entities` as an array, not as a nested `products` key) is an array of products — typically 20–50 entries with fields including `asin`, `title`, `price`, `currency`, `image_url`, `position`, `rating`, `amazons_choice`, and more, and a few more fields. We need to:

1. **Call the agent once per query row.** That's the `FROM KEYWORD_QUERIES q , TABLE(NIMBLE_AGENT_RUN('amazon_serp', OBJECT_CONSTRUCT('keyword', q.keyword))) a` lateral join. One UDTF call per input row.
2. **Explode each agent response into one row per product.** That's `LATERAL FLATTEN(INPUT => a.parsing) p` — Snowflake's tabular array unfold.
3. **Project typed columns out of each product object.** `p.value:asin::STRING`, `p.value:price::NUMBER(10, 2)`, etc.

The whole pipeline is one statement. No procedure, no Python, no `PARSE_JSON` — just SQL.

> **Confirm field names on first run.** The exact keys inside `parsing[*]` (`title`, `web_price`, `reviews_count`, …) are based on the [Nimble agent gallery](https://docs.nimbleway.com/nimble-sdk/agentic/agent-gallery) and can evolve. Before pinning this in production, run:
> ```sql
> SELECT raw FROM TABLE(NIMBLE_AGENT_RUN('amazon_serp', OBJECT_CONSTRUCT('keyword', 'noise canceling headphones')));
> ```
> and inspect the actual `data.parsing` shape against the projection below.

**Per-row error isolation.** If one query 429s, the UDTF yields a row with `status = 'http_429'` and `parsing IS NULL`. The `WHERE a.status = 'success'` filter below skips it — the rest of the queries still INSERT successfully.


In [ ]:
session.sql("""
INSERT INTO PRODUCT_SEARCH_RESULTS
    (keyword, category, position, asin, title, price, currency, rating,
     review_count, image_url, sponsored, status)
SELECT
    q.keyword                                          AS keyword,
    q.category                                       AS category,
    p.value:position::INTEGER                        AS position,
    p.value:asin::STRING                             AS asin,
    p.value:product_name::STRING                            AS title,
    p.value:price::NUMBER(10, 2)                 AS price,
    p.value:currency::STRING                         AS currency,
    p.value:rating::NUMBER(3, 2)                     AS rating,
    p.value:review_count::INTEGER                   AS review_count,
    p.value:image_url::STRING                            AS image_url,
    p.value:sponsored::BOOLEAN                       AS sponsored,
    a.status                                         AS status
FROM KEYWORD_QUERIES q,
     TABLE(NIMBLE_INTEGRATION.TOOLS.NIMBLE_AGENT_RUN(
         'amazon_serp',
         OBJECT_CONSTRUCT('keyword', q.keyword)
     )) a,
     LATERAL FLATTEN(INPUT => a.parsing) p
WHERE q.keyword IS NOT NULL
  AND a.status = 'success'
""").collect()

print("Enriched PRODUCT_SEARCH_RESULTS via NIMBLE_AGENT_RUN(amazon_serp) + LATERAL FLATTEN.")


In [ ]:
session.sql("""
SELECT keyword, position, asin, title, price, currency, rating, review_count, sponsored
FROM PRODUCT_SEARCH_RESULTS
QUALIFY ROW_NUMBER() OVER (PARTITION BY keyword ORDER BY position) <= 4
ORDER BY keyword, position
""").show()


SAMPLE OUTPUT — top 4 results per query (the actual run produces ~20-50 per query).
All values are illustrative placeholders.

+----------------------------+----------+-------------+---------------------------------------+--------+----------+--------+--------------+-----------+
| KEYWORD                    | POSITION | ASIN        | TITLE                                 | PRICE  | CURRENCY | RATING | REVIEW_COUNT | SPONSORED |
+----------------------------+----------+-------------+---------------------------------------+--------+----------+--------+--------------+-----------+
| kitchen knife set          |        1 | B0EXAMPLE21 | Brookline 14-Piece German Knife Set   | 149.99 | USD      |  4.70  |        12483 | TRUE      |
| kitchen knife set          |        2 | B0EXAMPLE22 | Northbrook Stainless 8-Piece Block    |  89.99 | USD      |  4.60  |         8214 | FALSE     |
| kitchen knife set          |        3 | B0EXAMPLE23 | Riverstone 6-Piece Chef Knife Set     |  64.97 | USD     

## From raw rankings to actionable signal

Daily ranking snapshots become valuable when they surface change — a new competitor entering the top results, a sponsored placement displacing an organic listing, a rating jump on a previously-unranked product. The view below flags one of those: **new entrants** — ASINs that appear in the top 20 for a tracked query for the first time in the last 7 days.

It's a window-function pattern: `MIN(enriched_at) OVER (PARTITION BY keyword, asin)` gives each ASIN's first-seen timestamp per query; rows in the latest snapshot whose first-seen is within the last 7 days are flagged.


In [ ]:
session.sql("""
CREATE OR REPLACE VIEW V_NEW_ENTRANTS AS
WITH first_seen AS (
    SELECT
        keyword,
        asin,
        MIN(enriched_at) AS first_seen_at,
        MAX(enriched_at) AS last_seen_at
    FROM PRODUCT_SEARCH_RESULTS
    WHERE status = 'success'
      AND position <= 20
    GROUP BY keyword, asin
),
latest_snapshot AS (
    SELECT keyword, MAX(enriched_at) AS snapshot_at
    FROM PRODUCT_SEARCH_RESULTS
    WHERE status = 'success'
    GROUP BY keyword
)
SELECT
    r.keyword,
    r.category,
    r.position,
    r.asin,
    r.title,
    r.price,
    r.rating,
    r.review_count,
    fs.first_seen_at,
    DATEDIFF('day', fs.first_seen_at, CURRENT_TIMESTAMP()) AS days_since_first_seen
FROM PRODUCT_SEARCH_RESULTS r
JOIN first_seen        fs ON fs.keyword = r.keyword AND fs.asin = r.asin
JOIN latest_snapshot   ls ON ls.keyword = r.keyword AND ls.snapshot_at = r.enriched_at
WHERE r.position <= 20
  AND fs.first_seen_at >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY r.keyword, r.position
""").collect()

print("V_NEW_ENTRANTS view created.")


In [ ]:
session.sql("""
SELECT keyword, position, asin, title, price, rating, review_count, days_since_first_seen
FROM V_NEW_ENTRANTS
ORDER BY keyword, position
""").show()


SAMPLE OUTPUT after ~10 days of daily runs. All values are illustrative placeholders.

+----------------------------+----------+-------------+-----------------------------------+--------+--------+--------------+-----------------------+
| KEYWORD                    | POSITION | ASIN        | TITLE                             | PRICE  | RATING | REVIEW_COUNT | DAYS_SINCE_FIRST_SEEN |
+----------------------------+----------+-------------+-----------------------------------+--------+--------+--------------+-----------------------+
| kitchen knife set          |        7 | B0EXAMPLE27 | Harborline 6-Piece Japanese Set   |  79.99 | USD    |  4.60        |                       3 |
| noise canceling headphones |        5 | B0EXAMPLE05 | Wexford ANC Wireless Over-Ear     | 129.99 | USD    |  4.50        |                       2 |
| noise canceling headphones |       12 | B0EXAMPLE08 | Cobalt Audio NC30 Budget          |  49.97 | USD    |  4.20        |                       4 |
+------------

## Now what

**Put it on a schedule.** [`schedule.sql`](./schedule.sql) wraps the lateral-join + FLATTEN `INSERT` in a Snowflake Task that runs daily at 08:00 Pacific. No stored proc needed — a Task can execute a single SQL statement directly.

**Wire signals into your downstream tools.** Pipe `V_NEW_ENTRANTS` into a Streamlit-in-Snowflake dashboard, a Slack webhook via a Snowflake External Function, or a row-change trigger on the underlying table.

**Track share-of-shelf per brand.** The next obvious analytic is brand-level — derive a `brand` from `title` (or a `BRAND_MAP` lookup table), then aggregate by `(query, brand, enriched_at)` to see who is gaining or losing ranking on each tracked query over time.

**Pair with `cpg_price_monitoring`.** Use this recipe to discover ASINs ranking on the queries you care about; feed the top-20 ASINs per query into the [`cpg_price_monitoring`](../cpg_price_monitoring/) PDP enrichment for daily per-ASIN price tracking. SERP finds the universe; PDP keeps it fresh.

**Swap in other marketplaces.** The Nimble [agent gallery](https://docs.nimbleway.com/nimble-sdk/agentic/agent-gallery) ships sibling SERP-style agents for other marketplaces (Walmart, Target, Best Buy, … — see the gallery for canonical names). Each one is a drop-in replacement for `'amazon_serp'` — the `parsing` shape changes per agent, so inspect with `SELECT raw FROM TABLE(NIMBLE_AGENT_RUN(<agent>, …))` once and update the projected columns.

**Handle failures.** Rows where `status` is `http_429`, `http_4xx`, `request_error`, or `error` are filtered out of `PRODUCT_SEARCH_RESULTS` by the `WHERE a.status = 'success'` clause. For observability, consider a parallel `PRODUCT_SEARCH_FAILURES` audit table that captures the non-success rows (query, status, raw error body) so you can spot rate-limit pressure or agent-side issues without staring at task logs.
